<a href="https://colab.research.google.com/github/Andrea-2215/BigData/blob/main/Activity_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

Saving MobileGames.csv to MobileGames.csv


{'MobileGames.csv': b'Game,Release date,As of,Player count[a],Publisher(s),Ref.,Table_Number\r\nMobile Legends: Bang Bang,December 2017,January 2017,150million peakdaily players,Moontoon,,1\r\nPUBG Mobile,March 2018,August 2023,300 millionmonthly players,Tencent Games/Krafton,,1\r\nCall of Duty: Mobile,"October 1, 2019",Nov 2024,1billion downloads[b],Activision,,1\r\nAmong Us,"June 15, 2018",November 2020,485million[c],InnerSloth,,1\r\nMini World,"December 26, 2015",April 2020,400million,Minovate,,1\r\nDragon Ball Z: Dokkan Battle,"January 30, 2015",August 2021,350million,Bandai Namco Entertainment,,1\r\nSonic Dash,"March 7, 2013",February 2020,350million,Sega,,1\r\nHelix Jump,"February 10, 2018",December 2018,334million,Voodoo,,1\r\nGardenscapes: New Acres,August 2016,May 2020,324million,Playrix,,1\r\nHomescapes,August 2017,May 2020,312million,Playrix,,1\r\nSuper Mario Run,"December 15, 2016",August 2018,300million,Nintendo,,1\r\nTownship,"February 24, 2012",May 2020,274million,Playri

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc

# LAB 4
# MEMBERS:
# ANDREA B. BOMBALES
# KATHLEEN ANN B. BORROMEO
# PAULEEN ANNE D. SIOJO

# Start Spark session
spark = SparkSession.builder \
    .appName("MobileGamesAnalysis") \
    .getOrCreate()

# Load CSV (infer schema first)
df = spark.read.csv(
    "/content/MobileGames.csv",
    header=True,
    inferSchema=True
)

# Rename columns
df = df.withColumnRenamed("Release date", "Release_Date") \
       .withColumnRenamed("As of", "As_Of") \
       .withColumnRenamed("Player count[a]", "Player_Count") \
       .withColumnRenamed("Publisher(s)", "Publisher") \

print("Schema after renaming:")
df.printSchema()

# Clean data

# Keep rows where Game and Publisher exist
clean_df = df.dropna(subset=["Game", "Publisher"])

# Remove exact duplicate rows
clean_df = clean_df.dropDuplicates()

print(f"Total rows after cleaning: {clean_df.count()}")
clean_df.show(5)

# Insight 1: Top publishers by number of games
top_publishers = clean_df.groupBy("Publisher") \
    .agg(count("*").alias("Total_Games")) \
    .orderBy(desc("Total_Games"))

print("\nINSIGHT 1: TOP PUBLISHERS")
top_publishers.show(10)

#Insight 2: Number of games released per date
games_per_date = clean_df.groupBy("Release_Date") \
    .agg(count("*").alias("Total_Games")) \
    .orderBy(desc("Total_Games"))

print("\nINSIGHT 2: GAMES RELEASED PER DATE")
games_per_date.show(10)

#  Insight 3: Most recent player count records
recent_records = clean_df.orderBy(desc("As_Of"))

print("\nINSIGHT 3: MOST RECENT PLAYER RECORDS")
recent_records.show(10)

# Save results as Parquet
clean_df.write.mode("overwrite").parquet("mobile_games_cleaned.parquet")
top_publishers.write.mode("overwrite").parquet("top_publishers.parquet")
games_per_date.write.mode("overwrite").parquet("games_per_date.parquet")
recent_records.write.mode("overwrite").parquet("recent_records.parquet")

print("Parquet files successfully saved.")

# Stop Spark
spark.stop()

Schema after renaming:
root
 |-- Game: string (nullable = true)
 |-- Release_Date: string (nullable = true)
 |-- As_Of: string (nullable = true)
 |-- Player_Count: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- Ref.: string (nullable = true)
 |-- Table_Number: integer (nullable = true)

Total rows after cleaning: 53
+-----------------+-----------------+-------------+------------+----------------+----+------------+
|             Game|     Release_Date|        As_Of|Player_Count|       Publisher|Ref.|Table_Number|
+-----------------+-----------------+-------------+------------+----------------+----+------------+
|     LINE Rangers|February 28, 2014|   March 2018|   50million|Line Corporation|NULL|           1|
|       Mini World|December 26, 2015|   April 2020|  400million|        Minovate|NULL|           1|
|White Cat Project|    July 14, 2014|    June 2016|  100million|          Colopl|NULL|           1|
|   World of Tanks|  August 12, 2010|   March 2016|  140mi